# 02 — AI-Assisted Vulnerability Discovery

**Context: Authorized security assessment only.** This notebook covers how AI tools accelerate vulnerability discovery in authorized penetration tests and CTF challenges.

## What
Vulnerability discovery with AI uses: (1) LLM-assisted code review to identify vulnerable patterns; (2) ML-guided fuzzing to prioritize interesting inputs; (3) AI-powered nuclei template generation; (4) semantic code search for dangerous function calls.

## Why
Manual code review of large codebases takes months. AI-assisted review can flag potential vulnerabilities in minutes, allowing human experts to focus on complex logical vulnerabilities. Bug bounty hunters and pentesters increasingly use AI to scale their efforts.

## Community context
GitHub Copilot and GPT-4 have been shown to find real CVEs in published code. Google Project Zero uses ML-assisted fuzzing (libFuzzer + ML-guided mutation). Meta's SapFix and Getafix apply ML to patch generation. The 2023 DARPA AIxCC (AI Cyber Challenge) specifically tests AI capability for vulnerability discovery and patching.

In [ ]:
# Vulnerable pattern detection: semantic analysis of code
# Simulates what an LLM or static analyzer would flag
import re
from dataclasses import dataclass
from typing import List

@dataclass
class Vulnerability:
    severity: str
    category: str
    line: int
    description: str
    cwe: str
    recommendation: str

class StaticAnalyzer:
    """
    Rule-based static analyzer that simulates what AI-enhanced SAST tools do.
    In production: these rules are learned/weighted by ML from historical findings.
    """

    PATTERNS = [
        # SQL Injection patterns
        {
            'regex': r'(?:execute|query)\s*\(.*\+.*(?:request|param|input|user)',
            'severity': 'CRITICAL',
            'category': 'SQL Injection',
            'cwe': 'CWE-89',
            'description': 'Possible SQL injection: user input concatenated into query',
            'recommendation': 'Use parameterised queries or prepared statements'
        },
        # Command injection
        {
            'regex': r'(?:os\.system|subprocess\.call|popen)\s*\(.*(?:\+|f[\'"])',
            'severity': 'CRITICAL',
            'category': 'Command Injection',
            'cwe': 'CWE-78',
            'description': 'Possible command injection: user-controlled string passed to shell',
            'recommendation': 'Use subprocess with list arguments, never string shell=True'
        },
        # Path traversal
        {
            'regex': r'open\s*\(.*(?:request|param|filename|path).*\)',
            'severity': 'HIGH',
            'category': 'Path Traversal',
            'cwe': 'CWE-22',
            'description': 'Possible path traversal: user-controlled file path',
            'recommendation': 'Validate and sanitise path, use os.path.abspath with allowlist'
        },
        # Hardcoded secrets
        {
            'regex': r'(?:password|secret|api_key|token)\s*=\s*[\'"][^\'"{][^\'"]{4,}[\'"]',
            'severity': 'HIGH',
            'category': 'Hardcoded Secret',
            'cwe': 'CWE-798',
            'description': 'Possible hardcoded credential',
            'recommendation': 'Use environment variables or secrets manager (AWS Secrets Manager, Vault)'
        },
        # Weak crypto
        {
            'regex': r'(?:md5|sha1|des)\s*\(',
            'severity': 'MEDIUM',
            'category': 'Weak Cryptography',
            'cwe': 'CWE-327',
            'description': 'Weak/deprecated cryptographic algorithm',
            'recommendation': 'Use SHA-256+, AES-256-GCM, or bcrypt/argon2 for passwords'
        },
    ]

    def analyze(self, code: str) -> List[Vulnerability]:
        findings = []
        lines = code.split('\n')
        for pattern in self.PATTERNS:
            regex = re.compile(pattern['regex'], re.IGNORECASE)
            for i, line in enumerate(lines, 1):
                if regex.search(line):
                    findings.append(Vulnerability(
                        severity=pattern['severity'],
                        category=pattern['category'],
                        line=i,
                        description=pattern['description'],
                        cwe=pattern['cwe'],
                        recommendation=pattern['recommendation']
                    ))
        return sorted(findings, key=lambda x: ['CRITICAL','HIGH','MEDIUM','LOW'].index(x.severity))

# Vulnerable Flask application example (intentionally vulnerable for training purposes)
vulnerable_code = '''
import os
from flask import Flask, request
import sqlite3
import hashlib

app = Flask(__name__)
db_password = "SuperSecret123"  # hardcoded credential
api_key = "sk-prod-abcdef123456"  # hardcoded API key

def get_user(username):
    conn = sqlite3.connect("users.db")
    # VULNERABLE: SQL injection
    query = "SELECT * FROM users WHERE name=" + username
    return conn.execute(query).fetchone()

def hash_password(password):
    return hashlib.md5(password.encode()).hexdigest()  # weak hash

@app.route("/file")
def read_file():
    filename = request.args.get("name")
    with open(filename, "r") as f:  # path traversal
        return f.read()

@app.route("/run")
def run_cmd():
    cmd = request.args.get("cmd")
    os.system("ls " + cmd)  # command injection
'''

analyzer = StaticAnalyzer()
findings = analyzer.analyze(vulnerable_code)
print(f'Vulnerability Scan Report ({len(findings)} findings):')
print('='*70)
for f in findings:
    print(f'  [{f.severity}] Line {f.line}: {f.category} ({f.cwe})')
    print(f'    Description: {f.description}')
    print(f'    Fix: {f.recommendation}')
    print()